# **Ingest **Races**.csv file**
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
     .Source Files 
     . Ingestion Time Stamp 
3. Write to bronze delta table      

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
source_file_name = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

#### **Step 1 - Read CSV file using the dataframe reader** 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DateType
races_schema = StructType([
    StructField('season', IntegerType(), True), 
    StructField('round', IntegerType(), True), 
    StructField('url', StringType(), True), 
    StructField('raceName', StringType(), True), 
    StructField('date', DateType(), True),     
    StructField('circuitId', StringType(), True),     
])


In [0]:
races_df = (
spark.read.format('csv') 
    .option('header','True')
  #  .option('inferSchema','True')
    .option('mode','FAILFAST')
    .schema(races_schema) 
    .load(source_file_name)
)

In [0]:
display(races_df)

### **Step 2 - Add Meta Data Columns**
- Source File
- Ingestions Time Stamp 

In [0]:
from pyspark.sql import functions as F
races_final_df = (
        races_df
        .withColumn('ingestion_timestamp', F.current_timestamp())
        .withColumn('source_file', F.col('_metadata.file_path'))
)        


In [0]:
display(races_final_df)

### **Step 3 : Write to bronze delta table**


In [0]:
(
    races_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('formula1.bronze.races')
)

In [0]:
%sql
select * from formula1.bronze.races

In [0]:
display(spark.table(table_name))